In [ ]:
# Credit Card Fraud Detection Using Machine Learning

## Notebook 04: Multiclass Classification

## 1. Multiclass Target and Dataset Verification

This section loads the processed training, validation, and test datasets generated in Notebook 2.

The target variable for multiclass classification is `fraud_type`. The class distribution will be examined in each dataset to verify which transaction categories are available for modelling.

In [1]:
import pandas as pd
from pathlib import Path

# Define the processed data folder
processed_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\Data\processed"
)

# Load the processed datasets
train_df = pd.read_csv(processed_folder / "train_processed.csv")
validation_df = pd.read_csv(processed_folder / "validation_processed.csv")
test_df = pd.read_csv(processed_folder / "test_processed.csv")

print("Dataset dimensions:")
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nAvailable columns:")
print("fraud_type present in training:", "fraud_type" in train_df.columns)
print("fraud_type present in validation:", "fraud_type" in validation_df.columns)
print("fraud_type present in test:", "fraud_type" in test_df.columns)

Dataset dimensions:
Training: (38476, 102)
Validation: (20141, 102)
Test: (59490, 102)

Available columns:
fraud_type present in training: True
fraud_type present in validation: True
fraud_type present in test: True


In [2]:
# Examine fraud_type distribution in each dataset

for name, data in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} - Fraud Type Distribution")
    print("-" * 50)

    print(data["fraud_type"].value_counts(dropna=False))

    print("\nClass percentages:")
    print(
        (data["fraud_type"].value_counts(normalize=True, dropna=False) * 100)
        .round(2)
    )


Training - Fraud Type Distribution
--------------------------------------------------
fraud_type
Legitimate            35416
Card-Not-Present        963
Account-Takeover        813
Synthetic-Identity      674
Card-Present            610
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            92.05
Card-Not-Present       2.50
Account-Takeover       2.11
Synthetic-Identity     1.75
Card-Present           1.59
Name: proportion, dtype: float64

Validation - Fraud Type Distribution
--------------------------------------------------
fraud_type
Legitimate            18540
Card-Not-Present        501
Account-Takeover        429
Card-Present            355
Synthetic-Identity      316
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            92.05
Card-Not-Present       2.49
Account-Takeover       2.13
Card-Present           1.76
Synthetic-Identity     1.57
Name: proportion, dtype: float64

Test - Fraud Type Distribution
------------------------------

In [3]:
# Verify consistency between fraud_type and is_fraud

for name, data in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} - Target Consistency")
    print("-" * 50)

    print(pd.crosstab(data["fraud_type"], data["is_fraud"]))

    expected_fraud = (data["fraud_type"] != "Legitimate").astype(int)

    mismatches = (expected_fraud != data["is_fraud"]).sum()

    print("\nNumber of mismatches:", mismatches)


Training - Target Consistency
--------------------------------------------------
is_fraud                0    1
fraud_type                    
Account-Takeover        0  813
Card-Not-Present        0  963
Card-Present            0  610
Legitimate          35416    0
Synthetic-Identity      0  674

Number of mismatches: 0

Validation - Target Consistency
--------------------------------------------------
is_fraud                0    1
fraud_type                    
Account-Takeover        0  429
Card-Not-Present        0  501
Card-Present            0  355
Legitimate          18540    0
Synthetic-Identity      0  316

Number of mismatches: 0

Test - Target Consistency
--------------------------------------------------
is_fraud                0     1
fraud_type                     
Account-Takeover        0  1329
Card-Not-Present        0  1415
Card-Present            0   933
Legitimate          54657     0
Synthetic-Identity      0  1156

Number of mismatches: 0


In [4]:
# Check data quality in all three datasets

for name, data in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} - Data Quality Check")
    print("-" * 50)

    print("Missing values:", data.isnull().sum().sum())
    print("Duplicate rows:", data.duplicated().sum())

    print("Missing fraud_type values:", data["fraud_type"].isnull().sum())


Training - Data Quality Check
--------------------------------------------------
Missing values: 0
Duplicate rows: 0
Missing fraud_type values: 0

Validation - Data Quality Check
--------------------------------------------------
Missing values: 0
Duplicate rows: 0
Missing fraud_type values: 0

Test - Data Quality Check
--------------------------------------------------
Missing values: 0
Duplicate rows: 0
Missing fraud_type values: 0


In [5]:
# Identify multiclass target and predictor columns

target_column = "fraud_type"

excluded_columns = [
    "fraud_type",
    "is_fraud",
    "source_row_index"
]

feature_columns = [
    column for column in train_df.columns
    if column not in excluded_columns
]

print("Target column:", target_column)
print("Excluded columns:", excluded_columns)

print("\nTotal predictor features:", len(feature_columns))

print("\nTarget classes:")
print(sorted(train_df[target_column].unique()))

print("\nFirst 15 predictor features:")
print(feature_columns[:15])

print("\nExcluded columns present in predictors:",
      set(excluded_columns).intersection(feature_columns))

Target column: fraud_type
Excluded columns: ['fraud_type', 'is_fraud', 'source_row_index']

Total predictor features: 99

Target classes:
['Account-Takeover', 'Card-Not-Present', 'Card-Present', 'Legitimate', 'Synthetic-Identity']

First 15 predictor features:
['binary__card_present', 'binary__international_transaction', 'binary__high_risk_country', 'binary__used_chip', 'binary__used_pin', 'binary__device_change_7d', 'binary__billing_shipping_mismatch', 'binary__account_profile_change_7d', 'binary__pos_entry_fallback', 'count_log__transactions_last_1h', 'count_log__transactions_last_24h', 'count_log__failed_auth_attempts_24h', 'count_log__merchant_velocity_24h', 'count_log__pin_retry_count_24h', 'count_original__num_cards_held']

Excluded columns present in predictors: set()


In [6]:
# Create predictors (X) and multiclass targets (y)

X_train = train_df[feature_columns].copy()
X_validation = validation_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[target_column].copy()
y_validation = validation_df[target_column].copy()
y_test = test_df[target_column].copy()

# Verify dataset dimensions
print("Feature dimensions:")
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nTarget dimensions:")
print("Training:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

# Verify feature-target alignment
print("\nFeature-target alignment:")
print("Training:", len(X_train) == len(y_train))
print("Validation:", len(X_validation) == len(y_validation))
print("Test:", len(X_test) == len(y_test))

Feature dimensions:
Training: (38476, 99)
Validation: (20141, 99)
Test: (59490, 99)

Target dimensions:
Training: (38476,)
Validation: (20141,)
Test: (59490,)

Feature-target alignment:
Training: True
Validation: True
Test: True


In [7]:
# Verify class distribution in the target variables

for name, target in [
    ("Training", y_train),
    ("Validation", y_validation),
    ("Test", y_test)
]:
    print(f"\n{name} - Target Distribution")
    print("-" * 50)

    print(target.value_counts())

    print("\nClass percentages:")
    print((target.value_counts(normalize=True) * 100).round(2))


Training - Target Distribution
--------------------------------------------------
fraud_type
Legitimate            35416
Card-Not-Present        963
Account-Takeover        813
Synthetic-Identity      674
Card-Present            610
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            92.05
Card-Not-Present       2.50
Account-Takeover       2.11
Synthetic-Identity     1.75
Card-Present           1.59
Name: proportion, dtype: float64

Validation - Target Distribution
--------------------------------------------------
fraud_type
Legitimate            18540
Card-Not-Present        501
Account-Takeover        429
Card-Present            355
Synthetic-Identity      316
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            92.05
Card-Not-Present       2.49
Account-Takeover       2.13
Card-Present           1.76
Synthetic-Identity     1.57
Name: proportion, dtype: float64

Test - Target Distribution
------------------------------------------

## 2. Multiclass Model Training Before SMOTE

In this section, three classification algorithms are trained using the original imbalanced training dataset:

1. Logistic Regression
2. Decision Tree
3. Random Forest

The models use the Expanded feature set containing 99 predictors.

Their performance will first be evaluated on the original validation dataset before applying SMOTE. This establishes the baseline performance for comparison with the models trained after handling class imbalance.

In [8]:
# Import evaluation metrics and classification algorithms

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Classification algorithms imported successfully.")
print("Evaluation metrics imported successfully.")

Classification algorithms imported successfully.
Evaluation metrics imported successfully.


In [9]:
# Initialize multiclass classification models

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Logistic Regression model initialized successfully.")
print("Decision Tree model initialized successfully.")
print("Random Forest model initialized successfully.")

Logistic Regression model initialized successfully.
Decision Tree model initialized successfully.
Random Forest model initialized successfully.


In [10]:
# Train Logistic Regression on the original training data

logistic_model.fit(X_train, y_train)

print("Multiclass Logistic Regression training completed.")

Multiclass Logistic Regression training completed.


In [11]:
# Generate Logistic Regression validation predictions

y_pred_logistic = logistic_model.predict(X_validation)

print("Logistic Regression validation predictions generated.")
print("Number of predictions:", len(y_pred_logistic))

Logistic Regression validation predictions generated.
Number of predictions: 20141


In [12]:
# Evaluate Multiclass Logistic Regression

logistic_accuracy = accuracy_score(y_validation, y_pred_logistic)

logistic_precision = precision_score(
    y_validation, y_pred_logistic, average="weighted", zero_division=0
)

logistic_recall = recall_score(
    y_validation, y_pred_logistic, average="weighted", zero_division=0
)

logistic_f1 = f1_score(
    y_validation, y_pred_logistic, average="weighted", zero_division=0
)

logistic_cm = confusion_matrix(y_validation, y_pred_logistic)

print("Multiclass Logistic Regression - Validation Evaluation")
print("-" * 55)
print(f"Accuracy:  {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall:    {logistic_recall:.4f}")
print(f"F1-score:  {logistic_f1:.4f}")

print("\nConfusion Matrix:")
print(logistic_cm)

Multiclass Logistic Regression - Validation Evaluation
-------------------------------------------------------
Accuracy:  0.9928
Precision: 0.9932
Recall:    0.9928
F1-score:  0.9926

Confusion Matrix:
[[  419     9     0     0     1]
 [    3   474     0     0    24]
 [    0     0   351     3     1]
 [    0     1     0 18539     0]
 [    0   101     1     1   213]]


In [13]:
# Train Decision Tree on the original training data

decision_tree_model.fit(X_train, y_train)

print("Multiclass Decision Tree training completed.")

Multiclass Decision Tree training completed.


In [14]:
# Generate Decision Tree validation predictions

y_pred_tree = decision_tree_model.predict(X_validation)

print("Decision Tree validation predictions generated.")
print("Number of predictions:", len(y_pred_tree))

Decision Tree validation predictions generated.
Number of predictions: 20141


In [15]:
# Evaluate Multiclass Decision Tree

tree_accuracy = accuracy_score(y_validation, y_pred_tree)

tree_precision = precision_score(
    y_validation, y_pred_tree, average="weighted", zero_division=0
)

tree_recall = recall_score(
    y_validation, y_pred_tree, average="weighted", zero_division=0
)

tree_f1 = f1_score(
    y_validation, y_pred_tree, average="weighted", zero_division=0
)

tree_cm = confusion_matrix(y_validation, y_pred_tree)

print("Multiclass Decision Tree - Validation Evaluation")
print("-" * 55)
print(f"Accuracy:  {tree_accuracy:.4f}")
print(f"Precision: {tree_precision:.4f}")
print(f"Recall:    {tree_recall:.4f}")
print(f"F1-score:  {tree_f1:.4f}")

print("\nConfusion Matrix:")
print(tree_cm)

Multiclass Decision Tree - Validation Evaluation
-------------------------------------------------------
Accuracy:  0.9885
Precision: 0.9887
Recall:    0.9885
F1-score:  0.9884

Confusion Matrix:
[[  399    22     5     0     3]
 [   11   445     0     1    44]
 [    1     1   345     3     5]
 [    0     7     5 18524     4]
 [    6   104     6     3   197]]


In [16]:
# Train Random Forest on the original training data

random_forest_model.fit(X_train, y_train)

print("Multiclass Random Forest training completed.")

Multiclass Random Forest training completed.


In [17]:
# Generate Random Forest validation predictions

y_pred_rf = random_forest_model.predict(X_validation)

print("Random Forest validation predictions generated.")
print("Number of predictions:", len(y_pred_rf))

Random Forest validation predictions generated.
Number of predictions: 20141


In [18]:
# Evaluate Multiclass Random Forest

rf_accuracy = accuracy_score(y_validation, y_pred_rf)

rf_precision = precision_score(
    y_validation, y_pred_rf, average="weighted", zero_division=0
)

rf_recall = recall_score(
    y_validation, y_pred_rf, average="weighted", zero_division=0
)

rf_f1 = f1_score(
    y_validation, y_pred_rf, average="weighted", zero_division=0
)

rf_cm = confusion_matrix(y_validation, y_pred_rf)

print("Multiclass Random Forest - Validation Evaluation")
print("-" * 55)
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-score:  {rf_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_cm)

Multiclass Random Forest - Validation Evaluation
-------------------------------------------------------
Accuracy:  0.9930
Precision: 0.9932
Recall:    0.9930
F1-score:  0.9928

Confusion Matrix:
[[  425     3     0     0     1]
 [    5   475     0     1    20]
 [    0     0   343    12     0]
 [    0     0     0 18540     0]
 [    3    92     1     3   217]]


In [19]:
# Compare multiclass models before SMOTE

multiclass_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        logistic_accuracy,
        tree_accuracy,
        rf_accuracy
    ],
    "Precision": [
        logistic_precision,
        tree_precision,
        rf_precision
    ],
    "Recall": [
        logistic_recall,
        tree_recall,
        rf_recall
    ],
    "F1_Score": [
        logistic_f1,
        tree_f1,
        rf_f1
    ]
})

display(
    multiclass_comparison.sort_values(
        by="F1_Score",
        ascending=False
    ).reset_index(drop=True)
)

,Model,Accuracy,Precision,Recall,F1_Score
0,Random Forest,0.992999,0.993231,0.992999,0.992770
1,Logistic Regression,0.992801,0.993173,0.992801,0.992618
2,Decision Tree,0.988531,0.988722,0.988531,0.988379


## 3. Handling Class Imbalance Using SMOTE

The training dataset contains five transaction categories with a significant class imbalance. The Legitimate category represents approximately 92% of the training data, while each fraud category represents a much smaller proportion.

To address this imbalance, the Synthetic Minority Over-sampling Technique (SMOTE) will be applied to the training dataset.

SMOTE will generate synthetic samples for the minority classes to balance the class distribution. The validation and test datasets will remain unchanged to ensure fair model evaluation.

The three classification algorithms will then be retrained using the resampled training data and evaluated on the original validation dataset.

In [20]:
# Inspect original training class distribution

print("Original Training Class Distribution")
print("-" * 45)

print(y_train.value_counts())

print("\nClass percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

Original Training Class Distribution
---------------------------------------------
fraud_type
Legitimate            35416
Card-Not-Present        963
Account-Takeover        813
Synthetic-Identity      674
Card-Present            610
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            92.05
Card-Not-Present       2.50
Account-Takeover       2.11
Synthetic-Identity     1.75
Card-Present           1.59
Name: proportion, dtype: float64


In [21]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only to the training dataset

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("SMOTE applied successfully.")
print("Original training shape:", X_train.shape)
print("Resampled training shape:", X_train_smote.shape)

SMOTE applied successfully.
Original training shape: (38476, 99)
Resampled training shape: (177080, 99)


In [22]:
# Verify class distribution after SMOTE

print("Resampled Training Class Distribution")
print("-" * 45)

print(y_train_smote.value_counts())

print("\nClass percentages:")
print(
    (y_train_smote.value_counts(normalize=True) * 100).round(2)
)

Resampled Training Class Distribution
---------------------------------------------
fraud_type
Legitimate            35416
Card-Present          35416
Synthetic-Identity    35416
Account-Takeover      35416
Card-Not-Present      35416
Name: count, dtype: int64

Class percentages:
fraud_type
Legitimate            20.0
Card-Present          20.0
Synthetic-Identity    20.0
Account-Takeover      20.0
Card-Not-Present      20.0
Name: proportion, dtype: float64


In [23]:
# Initialize Logistic Regression after SMOTE

logistic_smote_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Logistic Regression (SMOTE) model initialized successfully.")

Logistic Regression (SMOTE) model initialized successfully.


In [24]:
# Train Logistic Regression after SMOTE

logistic_smote_model.fit(
    X_train_smote,
    y_train_smote
)

print("Logistic Regression (SMOTE) training completed.")

Logistic Regression (SMOTE) training completed.


In [25]:
# Generate validation predictions after SMOTE

y_pred_logistic_smote = logistic_smote_model.predict(X_validation)

print("Logistic Regression (SMOTE) validation predictions generated.")

Logistic Regression (SMOTE) validation predictions generated.


In [26]:
# Evaluate Logistic Regression after SMOTE

logistic_smote_accuracy = accuracy_score(
    y_validation, y_pred_logistic_smote
)

logistic_smote_precision = precision_score(
    y_validation, y_pred_logistic_smote,
    average="weighted", zero_division=0
)

logistic_smote_recall = recall_score(
    y_validation, y_pred_logistic_smote,
    average="weighted", zero_division=0
)

logistic_smote_f1 = f1_score(
    y_validation, y_pred_logistic_smote,
    average="weighted", zero_division=0
)

logistic_smote_cm = confusion_matrix(
    y_validation, y_pred_logistic_smote
)

print("Logistic Regression (After SMOTE) Evaluation")
print("-" * 50)
print(f"Accuracy:  {logistic_smote_accuracy:.4f}")
print(f"Precision: {logistic_smote_precision:.4f}")
print(f"Recall:    {logistic_smote_recall:.4f}")
print(f"F1-score:  {logistic_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(logistic_smote_cm)

Logistic Regression (After SMOTE) Evaluation
--------------------------------------------------
Accuracy:  0.9916
Precision: 0.9919
Recall:    0.9916
F1-score:  0.9915

Confusion Matrix:
[[  415    11     1     0     2]
 [    4   465     0     0    32]
 [    0     0   354     1     0]
 [    0     3     8 18526     3]
 [    2   101     1     0   212]]


In [27]:
# Detailed classification report after SMOTE

print("Logistic Regression (After SMOTE) - Classification Report")
print("-" * 65)

print(
    classification_report(
        y_validation,
        y_pred_logistic_smote,
        zero_division=0
    )
)

Logistic Regression (After SMOTE) - Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.99      0.97      0.98       429
  Card-Not-Present       0.80      0.93      0.86       501
      Card-Present       0.97      1.00      0.98       355
        Legitimate       1.00      1.00      1.00     18540
Synthetic-Identity       0.85      0.67      0.75       316

          accuracy                           0.99     20141
         macro avg       0.92      0.91      0.91     20141
      weighted avg       0.99      0.99      0.99     20141



In [28]:
# Initialize Decision Tree after SMOTE

decision_tree_smote_model = DecisionTreeClassifier(
    random_state=42
)

print("Decision Tree (SMOTE) model initialized successfully.")

Decision Tree (SMOTE) model initialized successfully.


In [29]:
# Train Decision Tree after SMOTE

decision_tree_smote_model.fit(
    X_train_smote,
    y_train_smote
)

print("Decision Tree (SMOTE) training completed.")

Decision Tree (SMOTE) training completed.


In [30]:
# Generate Decision Tree validation predictions after SMOTE

y_pred_tree_smote = decision_tree_smote_model.predict(
    X_validation
)

print("Decision Tree (SMOTE) validation predictions generated.")

Decision Tree (SMOTE) validation predictions generated.


In [31]:
# Evaluate Decision Tree after SMOTE

tree_smote_accuracy = accuracy_score(
    y_validation, y_pred_tree_smote
)

tree_smote_precision = precision_score(
    y_validation, y_pred_tree_smote,
    average="weighted", zero_division=0
)

tree_smote_recall = recall_score(
    y_validation, y_pred_tree_smote,
    average="weighted", zero_division=0
)

tree_smote_f1 = f1_score(
    y_validation, y_pred_tree_smote,
    average="weighted", zero_division=0
)

tree_smote_cm = confusion_matrix(
    y_validation, y_pred_tree_smote
)

print("Decision Tree (After SMOTE) Evaluation")
print("-" * 50)
print(f"Accuracy:  {tree_smote_accuracy:.4f}")
print(f"Precision: {tree_smote_precision:.4f}")
print(f"Recall:    {tree_smote_recall:.4f}")
print(f"F1-score:  {tree_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(tree_smote_cm)

Decision Tree (After SMOTE) Evaluation
--------------------------------------------------
Accuracy:  0.9875
Precision: 0.9874
Recall:    0.9875
F1-score:  0.9874

Confusion Matrix:
[[  399    23     2     0     5]
 [   27   409     2     1    62]
 [    1     1   345     5     3]
 [    1     2     6 18528     3]
 [    9    93     4     1   209]]


In [32]:
# Detailed classification report after SMOTE

print("Decision Tree (After SMOTE) - Classification Report")
print("-" * 65)

print(
    classification_report(
        y_validation,
        y_pred_tree_smote,
        zero_division=0
    )
)

Decision Tree (After SMOTE) - Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.91      0.93      0.92       429
  Card-Not-Present       0.77      0.82      0.79       501
      Card-Present       0.96      0.97      0.97       355
        Legitimate       1.00      1.00      1.00     18540
Synthetic-Identity       0.74      0.66      0.70       316

          accuracy                           0.99     20141
         macro avg       0.88      0.88      0.88     20141
      weighted avg       0.99      0.99      0.99     20141



In [33]:
# Initialize Random Forest after SMOTE

random_forest_smote_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Random Forest (SMOTE) model initialized successfully.")

Random Forest (SMOTE) model initialized successfully.


In [34]:
# Train Random Forest after SMOTE

random_forest_smote_model.fit(
    X_train_smote,
    y_train_smote
)

print("Random Forest (SMOTE) training completed.")

Random Forest (SMOTE) training completed.


In [35]:
# Generate Random Forest validation predictions after SMOTE

y_pred_rf_smote = random_forest_smote_model.predict(
    X_validation
)

print("Random Forest (SMOTE) validation predictions generated.")

Random Forest (SMOTE) validation predictions generated.


In [36]:
# Evaluate Random Forest after SMOTE

rf_smote_accuracy = accuracy_score(
    y_validation, y_pred_rf_smote
)

rf_smote_precision = precision_score(
    y_validation, y_pred_rf_smote,
    average="weighted", zero_division=0
)

rf_smote_recall = recall_score(
    y_validation, y_pred_rf_smote,
    average="weighted", zero_division=0
)

rf_smote_f1 = f1_score(
    y_validation, y_pred_rf_smote,
    average="weighted", zero_division=0
)

rf_smote_cm = confusion_matrix(
    y_validation, y_pred_rf_smote
)

print("Random Forest (After SMOTE) Evaluation")
print("-" * 50)
print(f"Accuracy:  {rf_smote_accuracy:.4f}")
print(f"Precision: {rf_smote_precision:.4f}")
print(f"Recall:    {rf_smote_recall:.4f}")
print(f"F1-score:  {rf_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_smote_cm)

Random Forest (After SMOTE) Evaluation
--------------------------------------------------
Accuracy:  0.9931
Precision: 0.9934
Recall:    0.9931
F1-score:  0.9930

Confusion Matrix:
[[  420     5     2     0     2]
 [    8   471     0     2    20]
 [    0     0   352     3     0]
 [    0     0     0 18539     1]
 [    2    90     1     2   221]]


In [37]:
# Detailed classification report after SMOTE

print("Random Forest (After SMOTE) - Classification Report")
print("-" * 65)

print(
    classification_report(
        y_validation,
        y_pred_rf_smote,
        zero_division=0
    )
)

Random Forest (After SMOTE) - Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.98      0.98      0.98       429
  Card-Not-Present       0.83      0.94      0.88       501
      Card-Present       0.99      0.99      0.99       355
        Legitimate       1.00      1.00      1.00     18540
Synthetic-Identity       0.91      0.70      0.79       316

          accuracy                           0.99     20141
         macro avg       0.94      0.92      0.93     20141
      weighted avg       0.99      0.99      0.99     20141



In [38]:
# Compare multiclass models before and after SMOTE

multiclass_smote_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "SMOTE_Status": [
        "Before SMOTE",
        "Before SMOTE",
        "Before SMOTE",
        "After SMOTE",
        "After SMOTE",
        "After SMOTE"
    ],
    "Accuracy": [
        logistic_accuracy,
        tree_accuracy,
        rf_accuracy,
        logistic_smote_accuracy,
        tree_smote_accuracy,
        rf_smote_accuracy
    ],
    "Precision": [
        logistic_precision,
        tree_precision,
        rf_precision,
        logistic_smote_precision,
        tree_smote_precision,
        rf_smote_precision
    ],
    "Recall": [
        logistic_recall,
        tree_recall,
        rf_recall,
        logistic_smote_recall,
        tree_smote_recall,
        rf_smote_recall
    ],
    "F1_Score": [
        logistic_f1,
        tree_f1,
        rf_f1,
        logistic_smote_f1,
        tree_smote_f1,
        rf_smote_f1
    ]
})

display(
    multiclass_smote_comparison.sort_values(
        by="F1_Score",
        ascending=False
    ).reset_index(drop=True)
)

,Model,SMOTE_Status,Accuracy,Precision,Recall,F1_Score
0,Random Forest,After SMOTE,0.993148,0.993354,0.993148,0.992961
1,Random Forest,Before SMOTE,0.992999,0.993231,0.992999,0.992770
2,Logistic Regression,Before SMOTE,0.992801,0.993173,0.992801,0.992618
3,Logistic Regression,After SMOTE,0.991609,0.991899,0.991609,0.991467
4,Decision Tree,Before SMOTE,0.988531,0.988722,0.988531,0.988379
5,Decision Tree,After SMOTE,0.987538,0.987445,0.987538,0.987440


## 4. Final Multiclass Model Evaluation on Test Data

The six trained model configurations will be evaluated using the original test dataset.

The test dataset was not used during model training or SMOTE resampling. It provides a final evaluation of model performance on unseen transactions.

Accuracy, weighted Precision, weighted Recall, weighted F1-score, Confusion Matrix, and class-wise classification reports will be examined.

In [39]:
# Generate test predictions for all six models

y_test_pred_logistic = logistic_model.predict(X_test)
y_test_pred_tree = decision_tree_model.predict(X_test)
y_test_pred_rf = random_forest_model.predict(X_test)

y_test_pred_logistic_smote = logistic_smote_model.predict(X_test)
y_test_pred_tree_smote = decision_tree_smote_model.predict(X_test)
y_test_pred_rf_smote = random_forest_smote_model.predict(X_test)

print("Test predictions generated for all six multiclass models.")

Test predictions generated for all six multiclass models.


In [40]:
# Evaluate all six multiclass models on test data

test_predictions = {
    "Logistic Regression - Before SMOTE": y_test_pred_logistic,
    "Decision Tree - Before SMOTE": y_test_pred_tree,
    "Random Forest - Before SMOTE": y_test_pred_rf,
    "Logistic Regression - After SMOTE": y_test_pred_logistic_smote,
    "Decision Tree - After SMOTE": y_test_pred_tree_smote,
    "Random Forest - After SMOTE": y_test_pred_rf_smote
}

test_results = []

for model_name, predictions in test_predictions.items():

    test_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions,
            average="weighted", zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions,
            average="weighted", zero_division=0
        ),
        "F1_Score": f1_score(
            y_test, predictions,
            average="weighted", zero_division=0
        )
    })

multiclass_test_comparison = pd.DataFrame(test_results)

display(
    multiclass_test_comparison.sort_values(
        by="F1_Score",
        ascending=False
    ).reset_index(drop=True)
)

,Model,Accuracy,Precision,Recall,F1_Score
0,Random Forest - Before SMOTE,0.988132,0.988802,0.988132,0.987573
1,Random Forest - After SMOTE,0.987964,0.988585,0.987964,0.987462
2,Logistic Regression - Before SMOTE,0.986418,0.988099,0.986418,0.985677
3,Logistic Regression - After SMOTE,0.985241,0.986965,0.985241,0.984567
4,Decision Tree - After SMOTE,0.982888,0.983261,0.982888,0.982559
5,Decision Tree - Before SMOTE,0.982737,0.983171,0.982737,0.982341


In [41]:
# Random Forest Before SMOTE - Final Test Classification Report

print("Random Forest (Before SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_rf,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))

Random Forest (Before SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.97      0.98      0.98      1329
  Card-Not-Present       0.72      0.91      0.80      1415
      Card-Present       0.98      0.97      0.98       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.85      0.55      0.67      1156

          accuracy                           0.99     59490
         macro avg       0.91      0.88      0.89     59490
      weighted avg       0.99      0.99      0.99     59490


Confusion Matrix:
[[ 1303    19     1     0     6]
 [   22  1284     1     4   104]
 [    0     2   908    23     0]
 [    0     0     1 54656     0]
 [   12   473    14    24   633]]


In [42]:
# Random Forest After SMOTE - Final Test Classification Report

print("Random Forest (After SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_rf_smote,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf_smote))

Random Forest (After SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.97      0.97      0.97      1329
  Card-Not-Present       0.72      0.90      0.80      1415
      Card-Present       0.98      0.98      0.98       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.85      0.56      0.67      1156

          accuracy                           0.99     59490
         macro avg       0.90      0.88      0.88     59490
      weighted avg       0.99      0.99      0.99     59490


Confusion Matrix:
[[ 1295    25     1     0     8]
 [   31  1270     1     6   107]
 [    0     2   914    17     0]
 [    0     1     6 54650     0]
 [   13   463    12    23   645]]


In [43]:
# Logistic Regression Before SMOTE - Final Test Classification Report

print("Logistic Regression (Before SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_logistic,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_logistic))

Logistic Regression (Before SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.99      0.96      0.97      1329
  Card-Not-Present       0.67      0.93      0.78      1415
      Card-Present       0.98      0.98      0.98       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.84      0.46      0.60      1156

          accuracy                           0.99     59490
         macro avg       0.90      0.86      0.86     59490
      weighted avg       0.99      0.99      0.99     59490


Confusion Matrix:
[[ 1273    50     0     0     6]
 [   12  1309     2     0    92]
 [    0     2   913    17     1]
 [    0     1     4 54652     0]
 [    6   600    13     2   535]]


In [44]:
# Logistic Regression After SMOTE - Final Test Classification Report

print("Logistic Regression (After SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_logistic_smote,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_logistic_smote))

Logistic Regression (After SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.99      0.94      0.96      1329
  Card-Not-Present       0.66      0.92      0.77      1415
      Card-Present       0.94      0.99      0.96       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.82      0.46      0.59      1156

          accuracy                           0.99     59490
         macro avg       0.88      0.86      0.86     59490
      weighted avg       0.99      0.99      0.98     59490


Confusion Matrix:
[[ 1251    64     4     0    10]
 [   12  1305     3     0    95]
 [    0     1   924     8     0]
 [    0    11    35 54601    10]
 [    7   596    19     3   531]]


In [45]:
# Decision Tree Before SMOTE - Final Test Classification Report

print("Decision Tree (Before SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_tree,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_tree))

Decision Tree (Before SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.94      0.91      0.92      1329
  Card-Not-Present       0.66      0.83      0.73      1415
      Card-Present       0.93      0.97      0.95       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.72      0.50      0.59      1156

          accuracy                           0.98     59490
         macro avg       0.85      0.84      0.84     59490
      weighted avg       0.98      0.98      0.98     59490


Confusion Matrix:
[[ 1204    96     8     0    21]
 [   48  1174     5     3   185]
 [    5     4   907    12     5]
 [    0    12    27 54599    19]
 [   29   495    24    29   579]]


In [46]:
# Decision Tree After SMOTE - Final Test Classification Report

print("Decision Tree (After SMOTE) - Test Classification Report")
print("-" * 65)

print(
    classification_report(
        y_test,
        y_test_pred_tree_smote,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_tree_smote))

Decision Tree (After SMOTE) - Test Classification Report
-----------------------------------------------------------------
                    precision    recall  f1-score   support

  Account-Takeover       0.91      0.89      0.90      1329
  Card-Not-Present       0.66      0.81      0.73      1415
      Card-Present       0.94      0.97      0.96       933
        Legitimate       1.00      1.00      1.00     54657
Synthetic-Identity       0.74      0.53      0.62      1156

          accuracy                           0.98     59490
         macro avg       0.85      0.84      0.84     59490
      weighted avg       0.98      0.98      0.98     59490


Confusion Matrix:
[[ 1178   111     9     0    31]
 [   85  1143     8     5   174]
 [    5     0   908    18     2]
 [    0     3    19 54627     8]
 [   27   475    17    21   616]]


In [47]:
# Consolidated Multiclass Test Performance Comparison

test_comparison = []

for model_name, predictions in test_predictions.items():
    test_comparison.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),

        "Weighted_Precision": precision_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
        "Weighted_Recall": recall_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
        "Weighted_F1": f1_score(
            y_test, predictions, average="weighted", zero_division=0
        ),

        "Macro_Precision": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Macro_F1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        )
    })

multiclass_test_comparison = pd.DataFrame(test_comparison)

display(
    multiclass_test_comparison.sort_values(
        by="Macro_F1", ascending=False
    ).reset_index(drop=True)
)

,Model,Accuracy,Weighted_Precision,Weighted_Recall,Weighted_F1,Macro_Precision,Macro_Recall,Macro_F1
0,Random Forest - Before SMOTE,0.988132,0.988802,0.988132,0.987573,0.905874,0.881724,0.885068
1,Random Forest - After SMOTE,0.987964,0.988585,0.987964,0.987462,0.902950,0.881882,0.884483
2,Logistic Regression - Before SMOTE,0.986418,0.988099,0.986418,0.985677,0.895270,0.864845,0.864727
3,Logistic Regression - After SMOTE,0.985241,0.986965,0.985241,0.984567,0.880996,0.862448,0.856874
4,Decision Tree - After SMOTE,0.982888,0.983261,0.982888,0.982559,0.850981,0.839936,0.840489
5,Decision Tree - Before SMOTE,0.982737,0.983171,0.982737,0.982341,0.848880,0.841513,0.839324


In [48]:
# Save Final Multiclass Test Comparison

reports_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\reports"
)

reports_folder.mkdir(parents=True, exist_ok=True)

multiclass_test_comparison.to_csv(
    reports_folder / "multiclass_classification_test_comparison.csv",
    index=False
)

saved_file = reports_folder / "multiclass_classification_test_comparison.csv"

print("Final multiclass test comparison saved successfully.")
print("File exists:", saved_file.exists())
print("Number of model configurations:", len(multiclass_test_comparison))

Final multiclass test comparison saved successfully.
File exists: True
Number of model configurations: 6


## 5. Final Findings, Limitations and Conclusion

All six multiclass model configurations were evaluated on the original, untouched test dataset containing 59,490 transactions.

The evaluation included Accuracy, weighted Precision, weighted Recall, weighted F1-score, macro Precision, macro Recall, macro F1-score, Confusion Matrices, and class-wise Classification Reports.

**Key findings:**

1. Random Forest before SMOTE achieved the best overall test performance, with an accuracy of 98.81%, weighted F1-score of 98.76%, and macro F1-score of 88.51%.

2. Random Forest after SMOTE achieved a weighted F1-score of 98.75% and macro F1-score of 88.45%. Its Synthetic-Identity recall improved slightly from approximately 55% to 56%, but its overall performance was marginally lower.

3. Logistic Regression achieved a weighted F1-score of 98.57% before SMOTE and 98.46% after SMOTE. SMOTE did not improve its overall test performance or Synthetic-Identity recall.

4. Decision Tree achieved a weighted F1-score of 98.23% before SMOTE and 98.26% after SMOTE. Although SMOTE slightly improved its Synthetic-Identity recall, its performance on other fraud categories declined.

5. Random Forest before SMOTE was the strongest overall model across both weighted F1-score and macro F1-score. However, its Synthetic-Identity recall was only approximately 55%, indicating that this category remains difficult to identify.

6. The Legitimate category represents approximately 92% of the test dataset. Therefore, weighted metrics are strongly influenced by this majority category. Macro metrics and class-wise reports provide additional insight into minority-class performance.

**Conclusion:**

Random Forest before SMOTE was selected as the strongest overall multiclass model based on the final test evaluation. SMOTE did not produce a consistent improvement across the three algorithms. Its effect varied by model and fraud category.

**Limitations:**

- Friendly-Fraud transactions were excluded from the transaction-time modelling population because they were identified as post-transaction cases.
- Thirty-four expanded predictors had unresolved source or timing verification and were retained provisionally. Therefore, the exceptionally high performance should be treated as exploratory rather than proof of leakage-free predictive performance.
- Standard SMOTE was applied after one-hot encoding, which can generate fractional categorical indicator values.
- The test period was later than the training and validation periods, so the results also reflect temporal generalisation to later transactions.

The final six-model comparison was saved as `reports/multiclass_classification_test_comparison.csv`.

## 6. Saving and Verifying Multiclass Models and Reports

In [49]:
# Save All Six Trained Multiclass Models

import joblib
from pathlib import Path

models_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\models"
)

models_folder.mkdir(parents=True, exist_ok=True)

multiclass_models = {
    "logistic_regression_multiclass_before_smote.joblib": logistic_model,
    "decision_tree_multiclass_before_smote.joblib": decision_tree_model,
    "random_forest_multiclass_before_smote.joblib": random_forest_model,
    "logistic_regression_multiclass_after_smote.joblib": logistic_smote_model,
    "decision_tree_multiclass_after_smote.joblib": decision_tree_smote_model,
    "random_forest_multiclass_after_smote.joblib": random_forest_smote_model
}

for filename, model in multiclass_models.items():
    joblib.dump(model, models_folder / filename)

print("All six multiclass models saved successfully.")
print("Number of models saved:", len(multiclass_models))

All six multiclass models saved successfully.
Number of models saved: 6


In [50]:
# Verify Saved Multiclass Models

saved_multiclass_models = list(
    models_folder.glob("*_multiclass_*.joblib")
)

print("MULTICLASS MODEL VERIFICATION")
print("=" * 55)

for model_file in sorted(saved_multiclass_models):
    print(f"{model_file.name} — {model_file.stat().st_size / (1024 * 1024):.2f} MB")

print("-" * 55)
print("Total multiclass model files:", len(saved_multiclass_models))
print("All six models verified:", len(saved_multiclass_models) == 6)

MULTICLASS MODEL VERIFICATION
decision_tree_multiclass_after_smote.joblib — 0.13 MB
decision_tree_multiclass_before_smote.joblib — 0.05 MB
logistic_regression_multiclass_after_smote.joblib — 0.01 MB
logistic_regression_multiclass_before_smote.joblib — 0.01 MB
random_forest_multiclass_after_smote.joblib — 40.48 MB
random_forest_multiclass_before_smote.joblib — 13.21 MB
-------------------------------------------------------
Total multiclass model files: 6
All six models verified: True


In [51]:
# Verify Saved Models Can Be Reloaded

reloaded_multiclass_models = {}

for filename in multiclass_models:
    model_path = models_folder / filename
    loaded_model = joblib.load(model_path)

    reloaded_multiclass_models[filename] = loaded_model

    print(
        filename,
        "| Features:", loaded_model.n_features_in_,
        "| Classes:", len(loaded_model.classes_)
    )

print("-" * 55)
print("All six saved models loaded successfully:",
      len(reloaded_multiclass_models) == 6)

logistic_regression_multiclass_before_smote.joblib | Features: 99 | Classes: 5
decision_tree_multiclass_before_smote.joblib | Features: 99 | Classes: 5
random_forest_multiclass_before_smote.joblib | Features: 99 | Classes: 5
logistic_regression_multiclass_after_smote.joblib | Features: 99 | Classes: 5
decision_tree_multiclass_after_smote.joblib | Features: 99 | Classes: 5
random_forest_multiclass_after_smote.joblib | Features: 99 | Classes: 5
-------------------------------------------------------
All six saved models loaded successfully: True
